In [ ]:
import torch
import math
from transformers import GPT2Tokenizer, GPT2LMHeadModel, TextDataset, DataCollatorForLanguageModeling, Trainer, TrainingArguments


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"{device}")


model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

model = GPT2LMHeadModel.from_pretrained(model_name).to(device)
tokenizer.pad_token = tokenizer.eos_token


train_dataset = TextDataset(
    tokenizer=tokenizer,
    file_path="pretrain.txt",
    block_size=128

)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False
)

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="./tmp_baseline",
        per_device_eval_batch_size=8,
        fp16=True,
        report_to="none",
    ),
    data_collator=data_collator,
    eval_dataset=train_dataset,
)

eval_results = trainer.evaluate()
perplexity = math.exp(eval_results['eval_loss'])

print(f"Baseline Perplexity {perplexity:.2f}")

input_text = "In this study, we investigated the effect of"
inputs = tokenizer(input_text, return_tensors="pt").to(device)
input_ids = inputs["input_ids"]
attention_mask = inputs["attention_mask"]

print(f"Prompt: {input_text}")

output = model.generate(
    input_ids,
    attention_mask=inputs["attention_mask"],
    max_new_tokens=512,
    num_return_sequences=1,
    do_sample=True,
    top_k=50,
    top_p=0.95,
    pad_token_id=tokenizer.eos_token_id
) #Generating settings were suggested by AI to improve text quality.

generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(generated_text)